# Korea Valuation 통합 조회 v1

FCFF DCF · RIM · Relative Valuation 3개 모형의 DB 저장 결과를 통합 조회하고
upside 순위로 추출 → 엑셀(`C:\valuation results`) 저장하는 노트북.

| 함수 | 설명 |
|---|---|
| `get_valuation_dates()` | 모형별 valuation 수행 날짜 + 종목 수 조회 |
| `get_fcff_ranking()` | FCFF upside 순위 추출 |
| `get_rim_ranking()` | RIM upside 순위 추출 |
| `get_relative_ranking()` | 상대가치 upside 순위 추출 (PER/PBR/PSR 선택) |
| `get_combined_ranking()` | 3개 모형 통합 비교 순위 |

공통 파라미터:
- `n=50` : 상위 N개 (rank_range 미지정 시)
- `rank_range=(100, 150)` : 순위 구간 출력 (지정 시 n 무시)
- `dates=["2026-06-01", "2026-06-02"]` : 평가일 리스트. `None` → 최신 평가일 자동.
  여러 날 중복 종목은 **최신 측정일 기준** 1행만 사용.
- `save=True` : 엑셀 저장 여부 (기본 True). `file_format="csv"` 도 가능.
- `max_upside=None` : 데이터 품질용 upside 상한 필터 (예: 300 → +300% 초과 제외)


In [1]:
# ── Cell 1 · 경로 자동 감지 (기존 valuation 노트북과 동일) ──────
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",
]

def _setup_path() -> str:
    try:
        start = Path(__file__).resolve().parent
    except NameError:
        start = Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root 자동 감지 : {root}")
            return root
    for cand in _CANDIDATE_ROOTS:
        if os.path.isdir(cand) and os.path.isdir(os.path.join(cand, "DATA")):
            if cand not in sys.path:
                sys.path.insert(0, cand)
            print(f"[PATH] root 후보 경로 : {cand}")
            return cand
    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. _CANDIDATE_ROOTS 를 환경에 맞게 수정하세요."
    )

_ROOT = _setup_path()
print(f"[확인] 프로젝트 루트 : {_ROOT}")


[PATH] root 자동 감지 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] 프로젝트 루트 : C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy


In [2]:
# ── Cell 2 · Import & 설정 상수 ─────────────────────────────────
from datetime import datetime
from typing import Optional, List, Tuple, Union, Dict

import numpy as np
import pandas as pd
from sqlalchemy import text
from IPython.display import display

from DATA.config import get_db_info, get_engine
from DATA.korea_valuation_helpers import to_price_ticker

def log(tag: str, msg: str):
    ts = datetime.now().strftime("%H:%M:%S")
    print(f"[{ts}][{tag}] {msg}", flush=True)

# ══════════════════════════════════════════════════════════════
#  설정 — 여기만 수정하세요
# ══════════════════════════════════════════════════════════════

# 3개 valuation 결과 테이블 (각 노트북의 TABLE_RESULT 와 동일)
TABLE_FCFF = "korea_fcff_dcf_valuation_v7"   # FCFF DCF v7
TABLE_RIM  = "korea_rim_valuation"            # RIM
TABLE_REL  = "korea_relative_valuation"       # Relative (PBR/PSR/PER)

# 엑셀 저장 폴더
OUTPUT_DIR = r"C:\valuation results"

# 상대가치 지표명 ↔ DB 컬럼 suffix 매핑
#  ※ 상대가치 노트북은 PER/PBR/"PSR(주가매출비율)" 3종을 저장합니다.
#    PCR(주가현금흐름비율)은 DB에 없으므로 PSR 이 그 자리를 대신합니다.
REL_METRICS_ALL = ["PER", "PBR", "PSR"]

db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as c:
        c.execute(text("SELECT 1"))
    log("DB", f"연결 성공 host={db_info.get('host')} port={db_info.get('port')}")
except Exception as e:
    log("DB", f"연결 실패: {e}")

print(f"[설정] FCFF={TABLE_FCFF}  RIM={TABLE_RIM}  REL={TABLE_REL}")
print(f"[설정] 저장 폴더 = {OUTPUT_DIR}")


[12:19:57][DB] 연결 성공 host=192.168.0.230 port=3307
[설정] FCFF=korea_fcff_dcf_valuation_v7  RIM=korea_rim_valuation  REL=korea_relative_valuation
[설정] 저장 폴더 = C:\valuation results


In [3]:
# ── Cell 3 · 종목명/섹터 매핑 (FDR StockListing, 실패해도 진행) ──
NAME_LOOKUP: Dict[str, Dict[str, str]] = {}
try:
    import FinanceDataReader as fdr
    _listing = fdr.StockListing("KRX")
    _code_col = next((c for c in ["Code", "Symbol", "code", "ticker"]
                      if c in _listing.columns), None)
    _name_col = next((c for c in ["Name", "name", "기업명"]
                      if c in _listing.columns), None)
    _sec_col  = next((c for c in ["Sector", "Industry", "업종", "sector", "industry"]
                      if c in _listing.columns), None)
    if _code_col and _name_col:
        for _, row in _listing.iterrows():
            code = str(row[_code_col]).strip().zfill(6)
            NAME_LOOKUP[code] = {
                "name":   str(row[_name_col]) if pd.notna(row[_name_col]) else "",
                "sector": str(row[_sec_col])  if _sec_col and pd.notna(row.get(_sec_col)) else "",
            }
        log("NAME", f"FDR StockListing 로드 → {len(NAME_LOOKUP):,}개 종목명 매핑")
    else:
        log("NAME", f"[WARN] FDR 컬럼 불일치 → 종목명 공란 (cols={list(_listing.columns)})")
except Exception as e:
    log("NAME", f"[WARN] FDR StockListing 실패: {e} → 종목명 공란으로 진행")


def get_name_sector(ticker: str) -> Dict[str, str]:
    """'A005930' / '005930' → {'name': '삼성전자', 'sector': ...}"""
    try:
        code = to_price_ticker(ticker)
    except Exception:
        code = str(ticker).lstrip("A").zfill(6)
    return NAME_LOOKUP.get(code, {"name": "", "sector": ""})


[12:19:59][NAME] [WARN] FDR StockListing 실패: Expecting value: line 1 column 1 (char 0) → 종목명 공란으로 진행


In [4]:
# ── Cell 4 · 공통 유틸 (날짜 정규화 / 순위 슬라이스 / 파일 저장) ──

_MODEL_TABLE = {"FCFF": TABLE_FCFF, "RIM": TABLE_RIM, "Relative": TABLE_REL}


def get_valuation_dates(model: str = "all") -> pd.DataFrame:
    """valuation 수행 날짜 조회.

    Parameters
    ----------
    model : "fcff" | "rim" | "relative" | "all"

    Returns
    -------
    DataFrame [model, run_date, n_tickers, n_rows]  (최신순)
    """
    key = model.strip().lower()
    targets = (_MODEL_TABLE.items() if key == "all"
               else [(m, t) for m, t in _MODEL_TABLE.items() if m.lower() == key])
    if not targets:
        raise ValueError(f"model='{model}' 인식 불가. 'fcff'/'rim'/'relative'/'all' 중 선택.")

    frames = []
    for mname, tbl in targets:
        sql = text(f"""
            SELECT '{mname}' AS model, `date` AS run_date,
                   COUNT(DISTINCT ticker) AS n_tickers, COUNT(*) AS n_rows
            FROM `{tbl}`
            GROUP BY `date`
            ORDER BY `date` DESC
        """)
        try:
            df = pd.read_sql(sql, engine)
            frames.append(df)
        except Exception as e:
            log("DATES", f"[WARN] {mname}({tbl}) 조회 실패: {e}")
    if not frames:
        return pd.DataFrame(columns=["model", "run_date", "n_tickers", "n_rows"])
    out = pd.concat(frames, ignore_index=True)
    out["run_date"] = pd.to_datetime(out["run_date"]).dt.strftime("%Y-%m-%d")
    return out.sort_values(["model", "run_date"],
                           ascending=[True, False]).reset_index(drop=True)


def _resolve_dates(dates, table: str, model_name: str) -> List[str]:
    """입력 dates(None/str/list) → 해당 테이블에 실제 존재하는 날짜 리스트.

    - None / [] → 최신 평가일 1일 자동
    - 존재하지 않는 날짜는 경고 후 제외
    """
    avail = pd.read_sql(
        text(f"SELECT DISTINCT `date` FROM `{table}` ORDER BY `date` DESC"), engine)
    if avail.empty:
        raise ValueError(f"[{model_name}] {table} 에 저장된 결과가 없습니다.")
    avail_set = set(pd.to_datetime(avail["date"]).dt.strftime("%Y-%m-%d"))

    if dates is None or (isinstance(dates, (list, tuple)) and len(dates) == 0):
        latest = max(avail_set)
        log(model_name, f"날짜 미지정 → 최신 평가일 사용: {latest}")
        return [latest]

    if isinstance(dates, str):
        dates = [dates]
    norm = [pd.to_datetime(d).strftime("%Y-%m-%d") for d in dates]
    missing = sorted(set(norm) - avail_set)
    if missing:
        log(model_name, f"⚠️  측정 기록 없는 날짜 (무시): {missing}")
    valid = sorted(set(norm) & avail_set)
    if not valid:
        raise ValueError(f"[{model_name}] 입력한 날짜에 측정 기록이 없습니다. "
                         f"get_valuation_dates('{model_name.lower()}') 로 확인하세요.")
    return valid


def _slice_rank(df: pd.DataFrame, n: int,
                rank_range: Optional[Tuple[int, int]]) -> pd.DataFrame:
    """upside 내림차순 정렬 후 rank 부여 → 상위 n 또는 rank_range 구간."""
    df = df.reset_index(drop=True)
    df.insert(0, "rank", df.index + 1)
    if rank_range is not None:
        lo, hi = int(rank_range[0]), int(rank_range[1])
        if lo > hi:
            lo, hi = hi, lo
        out = df[(df["rank"] >= lo) & (df["rank"] <= hi)]
        if out.empty:
            log("RANK", f"⚠️  rank_range=({lo},{hi}) 구간에 종목 없음 "
                        f"(전체 {len(df)}개)")
        return out
    return df.head(int(n))


def _save_output(df: pd.DataFrame, method: str, run_dates: List[str],
                 save: bool = True, file_format: str = "xlsx") -> Optional[str]:
    """OUTPUT_DIR 에 {Method}_valuation_{수행날짜}_{출력날짜}.xlsx 형식으로 저장."""
    if not save:
        return None
    if df.empty:
        log("SAVE", "결과가 비어 있어 저장 생략")
        return None
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_tag = (run_dates[0] if len(run_dates) == 1
               else f"{min(run_dates)}_to_{max(run_dates)}")
    today = datetime.now().strftime("%Y-%m-%d")
    ext = "csv" if str(file_format).lower() == "csv" else "xlsx"
    fname = f"{method}_valuation_{run_tag}_{today}.{ext}"
    path = os.path.join(OUTPUT_DIR, fname)
    if ext == "csv":
        df.to_csv(path, index=False, encoding="utf-8-sig")
    else:
        df.to_excel(path, index=False)
    log("SAVE", f"저장 완료: {path}  ({len(df)}행)")
    return path


def _attach_name(df: pd.DataFrame, after_col: str = "ticker") -> pd.DataFrame:
    """ticker 다음 위치에 종목명 컬럼 삽입."""
    names = df["ticker"].map(lambda t: get_name_sector(t)["name"])
    pos = df.columns.get_loc(after_col) + 1
    df.insert(pos, "종목명", names)
    return df


In [5]:
# ── Cell 5 · FCFF DCF 순위 추출 ─────────────────────────────────

def get_fcff_ranking(n: int = 50,
                     rank_range: Optional[Tuple[int, int]] = None,
                     dates: Union[None, str, List[str]] = None,
                     save: bool = True,
                     file_format: str = "xlsx",
                     max_upside: Optional[float] = None) -> pd.DataFrame:
    """FCFF DCF 결과를 upside 내림차순 순위로 추출.

    Parameters
    ----------
    n          : 상위 N개 (rank_range 지정 시 무시)
    rank_range : (100, 150) 처럼 순위 구간 출력
    dates      : ["2026-06-01", "2026-06-02"] 평가일 리스트. None → 최신일.
                 여러 날 중복 종목은 최신 측정일 행만 사용.
    save       : True → C:\valuation results 에 엑셀 저장 (기본 True)
    file_format: "xlsx"(기본) | "csv"
    max_upside : upside 상한 필터(%). 예: 300 → +300% 초과 이상치 제외
    """
    run_dates = _resolve_dates(dates, TABLE_FCFF, "FCFF")
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}

    # 동일 ticker 가 여러 날 측정된 경우 최신 date 행 1개만 (행 내 정합성 유지)
    sql = text(f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date,
                target_price, current_price, upside_pct,
                discount_rate AS wacc, wacc_re, wacc_rd,
                g_terminal, moat_label, eva_spread,
                enterprise_value, equity_value, net_debt,
                nwc_method, revenue_quarters, forecast_model,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_FCFF}`
            WHERE `date` IN ({ph})
              AND target_price IS NOT NULL
              AND current_price > 0
              AND upside_pct IS NOT NULL
        ) t WHERE rn = 1
    """)
    raw = pd.read_sql(sql, engine, params=params).drop(columns=["rn"])
    if raw.empty:
        log("FCFF", "조건에 맞는 결과 없음")
        return raw
    if max_upside is not None:
        n_drop = int((raw["upside_pct"] > max_upside).sum())
        raw = raw[raw["upside_pct"] <= max_upside]
        if n_drop:
            log("FCFF", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_pct", ascending=False)

    out = pd.DataFrame({
        "ticker":        raw["ticker"],
        "측정일":         raw["measured_date"],
        "목표주가(원)":    raw["target_price"].round(0),
        "현재가(원)":      raw["current_price"].round(0),
        "Upside(%)":     raw["upside_pct"].round(1),
        "WACC(%)":       (raw["wacc"] * 100).round(2),
        "Re(%)":         (raw["wacc_re"] * 100).round(2),
        "Rd(%)":         (raw["wacc_rd"] * 100).round(2),
        "g_term(%)":     (raw["g_terminal"] * 100).round(2),
        "Moat":          raw["moat_label"],
        "EVA스프레드":     raw["eva_spread"].round(4),
        "기업가치EV(억원)": (raw["enterprise_value"] / 1e8).round(0),
        "순부채(억원)":    (raw["net_debt"] / 1e8).round(0),
        "NWC정의":        raw["nwc_method"],
        "매출분기수":      raw["revenue_quarters"],
        "예측모델":        raw["forecast_model"],
    })
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)

    print(f"\n{'='*90}")
    print(f"  [FCFF DCF] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*90}")
    display(out)
    _save_output(out, "FCFF", run_dates, save=save, file_format=file_format)
    return out


In [6]:
# ── Cell 6 · RIM 순위 추출 ──────────────────────────────────────

def get_rim_ranking(n: int = 50,
                    rank_range: Optional[Tuple[int, int]] = None,
                    dates: Union[None, str, List[str]] = None,
                    save: bool = True,
                    file_format: str = "xlsx",
                    max_upside: Optional[float] = None) -> pd.DataFrame:
    """RIM(잔여이익모형) 결과를 upside 내림차순 순위로 추출.

    파라미터는 get_fcff_ranking 과 동일.
    bv_source='IC_fallback' (음수 BV → IC 대체) 종목은 신뢰도 주의 컬럼으로 표시.
    """
    run_dates = _resolve_dates(dates, TABLE_RIM, "RIM")
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}

    # RIM 은 (ticker, date, year_label) 구조 → 종목·날짜별 다행.
    # TP/upside 등 요약값은 행마다 동일하게 저장되므로 최신 date 의 1행만 취함.
    sql = text(f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date,
                target_price, current_price, upside_pct,
                intrinsic_value, re, g_terminal, rho, n_phase2,
                moat_label, bv_source, beta_blume,
                revenue_quarters, forecast_model,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_RIM}`
            WHERE `date` IN ({ph})
              AND target_price IS NOT NULL
              AND current_price > 0
              AND upside_pct IS NOT NULL
        ) t WHERE rn = 1
    """)
    raw = pd.read_sql(sql, engine, params=params).drop(columns=["rn"])
    if raw.empty:
        log("RIM", "조건에 맞는 결과 없음")
        return raw
    if max_upside is not None:
        n_drop = int((raw["upside_pct"] > max_upside).sum())
        raw = raw[raw["upside_pct"] <= max_upside]
        if n_drop:
            log("RIM", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_pct", ascending=False)

    out = pd.DataFrame({
        "ticker":          raw["ticker"],
        "측정일":           raw["measured_date"],
        "목표주가(원)":      raw["target_price"].round(0),
        "현재가(원)":        raw["current_price"].round(0),
        "Upside(%)":       raw["upside_pct"].round(1),
        "내재가치(억원)":     (raw["intrinsic_value"] / 1e8).round(0),
        "Re(%)":           (raw["re"] * 100).round(2),
        "g_term(%)":       (raw["g_terminal"] * 100).round(2),
        "RI지속계수ω":      raw["rho"].round(3),
        "Phase2연수":       raw["n_phase2"],
        "Moat":            raw["moat_label"],
        "β_Blume":         raw["beta_blume"].round(3),
        "BV출처":           raw["bv_source"],      # IC_fallback = 신뢰도 주의
        "매출분기수":        raw["revenue_quarters"],
        "예측모델":          raw["forecast_model"],
    })
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)

    print(f"\n{'='*90}")
    print(f"  [RIM] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*90}")
    display(out)
    _save_output(out, "RIM", run_dates, save=save, file_format=file_format)
    return out


In [7]:
# ── Cell 7 · Relative Valuation 순위 추출 (PER/PBR/PSR 선택) ────

def get_relative_ranking(n: int = 50,
                         rank_range: Optional[Tuple[int, int]] = None,
                         dates: Union[None, str, List[str]] = None,
                         metrics: Union[str, List[str]] = "ALL",
                         save: bool = True,
                         file_format: str = "xlsx",
                         include_excluded: bool = False,
                         max_upside: Optional[float] = None) -> pd.DataFrame:
    """상대가치 결과를 upside 내림차순 순위로 추출.

    Parameters
    ----------
    metrics : "PER" | "PBR" | "PSR" | ["PER","PBR"] | "ALL"
        ※ DB 에는 PER/PBR/PSR 3종이 저장되어 있습니다 (PCR 미산출).
        - 1개 지정 → 해당 지표 upside 기준 정렬
        - 2개 이상/ALL → 선택 지표 upside 평균 기준 정렬
    include_excluded : True → 금융 등 제외 섹터(is_excluded=1)도 포함
    나머지 파라미터는 get_fcff_ranking 과 동일.
    """
    # ── 지표 정규화 ──
    if isinstance(metrics, str):
        metrics = REL_METRICS_ALL if metrics.strip().upper() == "ALL" else [metrics]
    mets = [m.strip().upper() for m in metrics]
    bad = [m for m in mets if m not in REL_METRICS_ALL]
    if bad:
        raise ValueError(
            f"지원하지 않는 지표 {bad}. 사용 가능: {REL_METRICS_ALL} "
            f"(PCR 은 상대가치 노트북에서 산출하지 않아 DB에 없습니다 — PSR 사용 권장)")

    run_dates = _resolve_dates(dates, TABLE_REL, "Relative")
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}
    excl_clause = "" if include_excluded else "AND is_excluded = 0"

    sql = text(f"""
        SELECT * FROM (
            SELECT
                ticker, `date` AS measured_date, sector, is_excluded,
                current_price,
                tp_per, tp_pbr, tp_psr, tp_avg,
                upside_per, upside_pbr, upside_psr, upside_avg,
                per_theory, pbr_theory, psr_theory,
                actual_per, actual_pbr, actual_psr,
                roe_y2, re_mid, g_est, eps_y2, bps_y2, npm_r2,
                ROW_NUMBER() OVER (
                    PARTITION BY ticker ORDER BY `date` DESC, id DESC
                ) AS rn
            FROM `{TABLE_REL}`
            WHERE `date` IN ({ph})
              AND current_price > 0
              {excl_clause}
        ) t WHERE rn = 1
    """)
    raw = pd.read_sql(sql, engine, params=params).drop(columns=["rn"])
    if raw.empty:
        log("REL", "조건에 맞는 결과 없음")
        return raw

    # ── 정렬 기준: 선택 지표 upside (1개=그 지표, 복수=평균) ──
    up_cols = [f"upside_{m.lower()}" for m in mets]
    raw["upside_sel"] = raw[up_cols].mean(axis=1, skipna=True)
    raw = raw[raw["upside_sel"].notna()]
    if max_upside is not None:
        n_drop = int((raw["upside_sel"] > max_upside).sum())
        raw = raw[raw["upside_sel"] <= max_upside]
        if n_drop:
            log("REL", f"upside > {max_upside}% 이상치 {n_drop}개 제외")

    raw["measured_date"] = pd.to_datetime(raw["measured_date"]).dt.strftime("%Y-%m-%d")
    raw = raw.sort_values("upside_sel", ascending=False)

    sel_label = "+".join(mets)
    cols = {
        "ticker":      raw["ticker"],
        "측정일":       raw["measured_date"],
        "섹터":         raw["sector"],
        "현재가(원)":    raw["current_price"].round(0),
        f"Upside_{sel_label}(%)": raw["upside_sel"].round(1),
    }
    for m in mets:  # 지표별 적정가/upside/이론·실제 멀티플
        lm = m.lower()
        cols[f"TP_{m}(원)"]    = raw[f"tp_{lm}"].round(0)
        cols[f"Upside_{m}(%)"] = raw[f"upside_{lm}"].round(1)
        cols[f"{m}_이론"]      = raw[f"{lm}_theory"].round(2)
        cols[f"{m}_실제"]      = raw[f"actual_{lm}"].round(2)
    cols.update({
        "ROE_y2(%)":  (raw["roe_y2"] * 100).round(2),
        "Re_mid(%)":  (raw["re_mid"] * 100).round(2),
        "g_est(%)":   (raw["g_est"] * 100).round(2),
        "EPS_y2(원)":  raw["eps_y2"].round(0),
        "BPS_y2(원)":  raw["bps_y2"].round(0),
        "NPM_R2":     raw["npm_r2"].round(3),
    })
    if include_excluded:
        cols["제외섹터"] = raw["is_excluded"]
    out = pd.DataFrame(cols)
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)

    print(f"\n{'='*90}")
    print(f"  [Relative {sel_label}] upside 순위  |  평가일 {run_dates}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*90}")
    display(out)
    _save_output(out, f"Relative_{sel_label}", run_dates,
                 save=save, file_format=file_format)
    return out


In [8]:
# ── Cell 8 · 3개 모형 통합 비교 순위 ────────────────────────────

def _fetch_model_upside(table: str, model_name: str,
                        dates) -> pd.DataFrame:
    """모형별 (ticker → 최신 측정일 TP/현재가/upside) 공통 추출."""
    run_dates = _resolve_dates(dates, table, model_name)
    ph = ", ".join(f":d{i}" for i in range(len(run_dates)))
    params = {f"d{i}": d for i, d in enumerate(run_dates)}

    if table == TABLE_REL:
        tp_expr, up_expr = "tp_avg", "upside_avg"
        extra_filter = "AND is_excluded = 0 AND tp_avg IS NOT NULL"
    else:
        tp_expr, up_expr = "target_price", "upside_pct"
        extra_filter = "AND target_price IS NOT NULL AND upside_pct IS NOT NULL"

    sql = text(f"""
        SELECT ticker, measured_date, tp, cp, upside FROM (
            SELECT ticker, `date` AS measured_date,
                   {tp_expr} AS tp, current_price AS cp, {up_expr} AS upside,
                   ROW_NUMBER() OVER (
                       PARTITION BY ticker ORDER BY `date` DESC, id DESC
                   ) AS rn
            FROM `{table}`
            WHERE `date` IN ({ph})
              AND current_price > 0
              {extra_filter}
        ) t WHERE rn = 1
    """)
    df = pd.read_sql(sql, engine, params=params)
    df["measured_date"] = pd.to_datetime(df["measured_date"]).dt.strftime("%Y-%m-%d")
    df.attrs["run_dates"] = run_dates
    return df


def get_combined_ranking(n: int = 50,
                         rank_range: Optional[Tuple[int, int]] = None,
                         dates: Union[None, str, List[str]] = None,
                         save: bool = True,
                         file_format: str = "xlsx",
                         min_models: int = 1,
                         max_upside: Optional[float] = None) -> pd.DataFrame:
    """FCFF · RIM · Relative(평균) 3개 모형 upside 를 종목별로 병합한 통합 순위.

    - 정렬 기준: 가용 모형 upside 의 단순평균 (Upside_평균)
    - min_models : 최소 몇 개 모형에서 평가된 종목만 포함 (기본 1, 보수적으로 보려면 3)
    - dates=None → 모형별 각각의 최신 평가일 사용
      (상대가치는 제외섹터 비포함 · tp_avg 기준)
    """
    parts = {}
    for label, tbl in [("FCFF", TABLE_FCFF), ("RIM", TABLE_RIM),
                       ("REL", TABLE_REL)]:
        try:
            mname = {"FCFF": "FCFF", "RIM": "RIM", "REL": "Relative"}[label]
            parts[label] = _fetch_model_upside(tbl, mname, dates)
        except ValueError as e:
            log("COMBINED", f"[WARN] {label} 건너뜀: {e}")

    if not parts:
        log("COMBINED", "사용 가능한 모형 결과가 없습니다.")
        return pd.DataFrame()

    merged = None
    all_run_dates = []
    for label, df in parts.items():
        all_run_dates += df.attrs.get("run_dates", [])
        sub = df.rename(columns={
            "tp": f"TP_{label}(원)", "upside": f"Upside_{label}(%)",
            "cp": f"_cp_{label}", "measured_date": f"_dt_{label}"})
        merged = sub if merged is None else merged.merge(sub, on="ticker", how="outer")

    up_cols = [c for c in merged.columns if c.startswith("Upside_")]
    cp_cols = [c for c in merged.columns if c.startswith("_cp_")]
    dt_cols = [c for c in merged.columns if c.startswith("_dt_")]

    merged["평가모형수"] = merged[up_cols].notna().sum(axis=1)
    merged = merged[merged["평가모형수"] >= int(min_models)]
    merged["Upside_평균(%)"] = merged[up_cols].mean(axis=1, skipna=True)
    if max_upside is not None:
        merged = merged[merged["Upside_평균(%)"] <= max_upside]
    merged["현재가(원)"] = merged[cp_cols].bfill(axis=1).iloc[:, 0]
    merged["최신측정일"] = merged[dt_cols].max(axis=1)

    merged = merged.sort_values("Upside_평균(%)", ascending=False)

    ordered = (["ticker", "최신측정일", "현재가(원)", "Upside_평균(%)", "평가모형수"]
               + sorted(up_cols)
               + sorted(c for c in merged.columns if c.startswith("TP_")))
    out = merged[ordered].copy()
    out["현재가(원)"] = out["현재가(원)"].round(0)
    for c in out.columns:
        if c.startswith("Upside"):
            out[c] = out[c].round(1)
        elif c.startswith("TP_"):
            out[c] = out[c].round(0)
    out = _attach_name(out)
    out = _slice_rank(out, n, rank_range)

    run_dates = sorted(set(all_run_dates))
    print(f"\n{'='*100}")
    print(f"  [통합 Combined] FCFF·RIM·Relative upside 평균 순위  |  "
          f"평가일 {run_dates}  |  min_models={min_models}  |  "
          f"{'rank ' + str(rank_range) if rank_range else 'TOP ' + str(n)}")
    print(f"{'='*100}")
    display(out)
    _save_output(out, "Combined", run_dates, save=save, file_format=file_format)
    return out


In [9]:
# ── Cell 9 · 0단계: valuation 수행 날짜부터 확인 ────────────────
dates_df = get_valuation_dates("all")   # "fcff" / "rim" / "relative" 개별 조회 가능
display(dates_df)


,model,run_date,n_tickers,n_rows
0,FCFF,2026-06-01,1266,10128
1,Relative,2026-04-26,7,7


## 사용 예시

```python
# 1) FCFF 상위 30개, 최신 평가일, 엑셀 저장(기본 True)
fcff_top = get_fcff_ranking(n=30)

# 2) RIM 100~150위 구간, 특정 날짜 2일 (중복 종목은 최신 측정일 우선)
rim_mid = get_rim_ranking(rank_range=(100, 150),
                          dates=["2026-06-01", "2026-06-02"])

# 3) 상대가치 — PER 단독 / PBR+PSR 조합 / 전체
rel_per = get_relative_ranking(n=50, metrics="PER")
rel_mix = get_relative_ranking(n=50, metrics=["PBR", "PSR"])
rel_all = get_relative_ranking(n=50, metrics="ALL", save=False)   # 저장 생략

# 4) 통합 — 3개 모형 모두 평가된 종목만, upside 평균 상위 50
combo = get_combined_ranking(n=50, min_models=3)

# 5) 데이터 품질 필터 — upside +300% 초과 이상치 제외
fcff_clean = get_fcff_ranking(n=100, max_upside=300)
```

저장 파일명 형식: `C:\valuation results\FCFF_valuation_2026-06-01_to_2026-06-02_2026-06-05.xlsx`
(= `{모형}_valuation_{수행날짜}_{출력날짜}.xlsx`)


In [11]:
# ── Cell 10 · 실행 예시 (필요 시 주석 해제) ─────────────────────
# fcff_top = get_fcff_ranking(n=30)
# rim_top  = get_rim_ranking(n=30)
# rel_top  = get_relative_ranking(n=30, metrics="ALL")
# combo    = get_combined_ranking(n=30, min_models=2)
fcff_range = get_fcff_ranking(rank_range=(300, 400), dates=["2026-06-01"])


  [FCFF DCF] upside 순위  |  평가일 ['2026-06-01']  |  rank (300, 400)


,rank,ticker,종목명,측정일,목표주가(원),현재가(원),Upside(%),WACC(%),Re(%),Rd(%),g_term(%),Moat,EVA스프레드,기업가치EV(억원),순부채(억원),NWC정의,매출분기수,예측모델
299,300,A101160,,2026-06-01,49462.0,26150.0,89.1,11.87,11.87,1.00,4.0,Narrow moat,0.1426,8167.0,0.0,legacy_proxy,26,Ensemble
300,301,A192440,,2026-06-01,60865.0,32200.0,89.0,9.94,9.94,1.65,4.0,No moat,-0.0408,3533.0,0.0,legacy_proxy,26,Ensemble
301,302,A086450,,2026-06-01,37887.0,20150.0,88.0,9.71,9.71,1.02,4.0,No moat,0.0076,17091.0,0.0,legacy_proxy,26,Ensemble
302,303,A264450,,2026-06-01,24881.0,13240.0,87.9,10.00,10.00,0.87,4.0,Some moat,0.0587,3649.0,0.0,legacy_proxy,26,Ensemble
303,304,A005850,,2026-06-01,141655.0,75600.0,87.4,10.86,10.86,0.76,4.0,No moat,0.0149,65240.0,0.0,legacy_proxy,65,Ensemble
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
395,396,A122870,,2026-06-01,60976.0,45750.0,33.3,9.99,9.99,0.91,4.0,No moat,-0.0002,11309.0,0.0,legacy_proxy,26,Ensemble
396,397,A000890,,2026-06-01,1946.0,1463.0,33.0,9.27,9.27,1.08,4.0,No moat,-0.0636,516.0,0.0,legacy_proxy,66,Ensemble
397,398,A032350,,2026-06-01,25875.0,19470.0,32.9,9.94,9.94,1.95,4.0,No moat,0.0246,20596.0,0.0,legacy_proxy,66,Ensemble
398,399,A036890,,2026-06-01,19462.0,14700.0,32.4,11.24,11.24,1.04,4.0,No moat,0.0032,3935.0,0.0,legacy_proxy,26,Ensemble


[12:23:44][SAVE] 저장 완료: C:\valuation results\FCFF_valuation_2026-06-01_2026-06-05.xlsx  (101행)
